# Content embeddings prep — filter Yambda audio embeddings to our vocab

One-time job: download the global `embeddings.parquet` (**13.8 GB**, 7.72M tracks), keep only our 629k train items, and save a compact aligned matrix (~300 MB). **Save the output as a Kaggle Dataset** so the two-tower / hybrid notebooks load it instantly and we never re-download 13.8 GB.

**Settings:** Internet On, GPU not needed. `HF_TOKEN` secret recommended (faster, no rate limit on a 13.8 GB pull).

In [ ]:
!pip install -q --upgrade "git+https://github.com/ak1232320/nndl_capstone_2026.git"

In [ ]:
# Set the data dir to the 20 GB working disk BEFORE importing ymrec (config reads it at import).
import os
os.environ["YMREC_DATA"] = "/kaggle/working/data"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception:
    print("No HF_TOKEN secret — anonymous works but a 13.8 GB pull may be rate-limited.")

In [ ]:
import time, numpy as np
from ymrec.data.prep import item_vocab
from ymrec.data.embeddings import load_aligned_embeddings

item_ids = item_vocab(size="50m")
print(f"vocab items: {len(item_ids):,}")

t0 = time.time()
emb, mask, dim = load_aligned_embeddings(item_ids)  # downloads 13.8 GB on first call
print(f"done in {(time.time()-t0)/60:.1f} min | dim={dim} | "
      f"coverage: {mask.mean():.3f} ({mask.sum():,}/{len(mask):,} items have audio)")

In [ ]:
out = "/kaggle/working"
np.save(f"{out}/content_emb.npy", emb)        # (n_items, dim) float32, aligned to item_ids
np.save(f"{out}/content_mask.npy", mask)      # (n_items,) bool — has audio embedding
np.save(f"{out}/item_ids.npy", item_ids)      # vocab, for an alignment assert downstream
print("saved:", emb.shape, emb.dtype, "->", out)
print("\nNEXT: Save Version, then create a Kaggle Dataset from these 3 .npy files")
print("named e.g. 'yambda-50m-content-emb'. The two-tower notebook will attach it.")